# Benchmark 1: 1 mJ, 200 fs, argon-filled Herriott cell

**Paper:** C. Grebing, M. Müller, J. Buldt, H. Stark, J. Limpert, *Kilowatt-average-power compression of millijoule pulses in a gas-filled multi-pass cell*, Opt. Lett. **45**, 6250 (2020), [doi:10.1364/OL.408998](https://doi.org/10.1364/OL.408998), open access at [arXiv:2103.00893](https://arxiv.org/abs/2103.00893).

**Why this case:** a clean, textbook SPM stage in the mJ range, with a moderate peak power (about 4.5 GW, P/Pc ≈ 0.2) and dielectric mirrors. The same group published the input of their own simulation of this stage (Müller *et al.*, Opt. Lett. 2021, [arXiv:2201.03276](https://arxiv.org/abs/2201.03276)): a Gaussian beam with a Gaussian pulse of 994 µJ and 200 fs, plus a TOD of −10⁴ fs³. That removes most of the guesswork on the input.

| | paper | used here |
|---|---|---|
| mirrors | RoC 0.6 m, 75 mm, R > 99.95 % and \|GDD\| < 50 fs² over 970–1130 nm | R = 0.9995, GDD = 0, band edges 10 nm outside the specified band (assumption) |
| geometry | d = 1183 mm, 26 focal passes, re-entrant | `herriott(R=0.6, N=26, k=1)`, d = 1182.6 mm |
| eigenmode | 0.30 mm diameter at focus, 2.5 mm on mirror | computed below |
| gas | Ar, 700 mbar, n₂ = 1×10⁻²³ m²/W at 1 atm | same |
| input | ~1 mJ, 200 fs, 1030 nm | 994 µJ, 200 fs, TOD −10⁴ fs³ (Müller 2021) |
| injection | mode-matched to the linear eigenmode | Gaussian at the linear eigenmode waist |
| compressor | 6 chirped mirrors, total −1923 fs² | same GDD |

**Measured results to compare with:** 20-dB width ≈ 120 nm, Fourier limit 27 fs, compressed 31 fs (autocorrelation), 96 % transmission (cell + compressor), M² < 1.15, intensity-weighted homogeneity 98.7 % (x) and 99.4 % (y).

**Output for benchmark 2:** the compressed output is saved to `bench1_stage1_output.npz`, the input of the second cell of Müller *et al.* (notebook 2).

In [ ]:
import os, time, json, hashlib
import numpy as np
import matplotlib.pyplot as plt
import thunder4d as th
from thunder4d import diagnostics as dg

FS = 1e-15
plt.rcParams.update({"figure.dpi": 110})
print("thunder4d", th.__version__, "| GPU available:", th.gpu_available())

In [ ]:
# ---------------- run settings ----------------
QUICK = False          # True: 2 passes on a coarse grid, only to check that the notebook runs
DEVICE = "auto"        # "auto" (GPU if CuPy works), "gpu" or "cpu"
CPU_BUDGET_GB = 16     # memory the run may use when it runs on the CPU
PHI_MAX = 0.02         # peak nonlinear phase per step: halve it once to check convergence
MAX_SECONDS = None     # e.g. 3600: stop cleanly after ~1 h, re-run the run cell to continue
RESUME = True          # continue from the checkpoint of an identical run if there is one

In [ ]:
# ---------------- helpers (same in the three benchmark notebooks) ----------------
def device_budget_gb():
    """Memory the run may use: 80 % of the free GPU memory, or CPU_BUDGET_GB on the CPU."""
    if DEVICE != "cpu" and th.gpu_available():
        import cupy as cp
        free, total = cp.cuda.runtime.memGetInfo()
        print(f"GPU: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")
        return 0.8 * free / 1e9
    print(f"CPU run, budget {CPU_BUDGET_GB} GB")
    return CPU_BUDGET_GB


def pick_grid(cell, presets, budget_gb, n_fields=12, **grid_args):
    """Finest preset whose estimated memory (about n_fields x one field) fits in budget_gb."""
    for p in presets:
        g = th.suggest_grid(cell, **grid_args, **p, device="cpu")       # axes only: cheap
        need = n_fields * np.prod(g.shape) * 8 / 1e9                    # complex64
        ok = need <= budget_gb
        print(f"  {p}: {g.Nx}x{g.Ny}x{g.Nt}, ~{need:.1f} GB" + ("  <- used" if ok else ""))
        if ok:
            return th.suggest_grid(cell, **grid_args, **p, device=DEVICE), p
    raise MemoryError("no preset fits the memory budget: add a coarser preset or use a bigger GPU")


def band_mirror(R_in, lam_lo, lam_hi, R_out=0.0, order=80):
    """Intensity reflectivity R(lambda) [lambda in m]: R_in inside the band, smooth edges with their
    50 % points at lam_lo and lam_hi, R_out far outside."""
    lc, hw = 0.5 * (lam_lo + lam_hi), 0.5 * (lam_hi - lam_lo)
    def R(lam):
        u = np.abs((np.asarray(lam, float) - lc) / hw)
        return R_out + (R_in - R_out) * np.exp(-np.log(2) * u ** order)
    return R


def gdd_phase(grid, gdd_fs2=0.0, tod_fs3=0.0):
    """Spectral phase on grid.Omega (FFT order) of a GDD [fs^2] and TOD [fs^3]."""
    W = grid.Omega * FS
    return gdd_fs2 / 2 * W ** 2 + tod_fs3 / 6 * W ** 3


def beam_pixels(Aw, rel=1e-4):
    """(npix, Nt) host array of the pixels that carry the beam (fluence > rel x max)."""
    A = th.to_host(Aw)
    F = np.sum(np.abs(A) ** 2, axis=2)
    return A[F > rel * F.max()]


def _pad(E, up):
    """Zero-pad spectra (FFT order, last axis) by `up` -> time step dt/up."""
    if up == 1:
        return E
    N = E.shape[-1]
    h = (N + 1) // 2
    out = np.zeros(E.shape[:-1] + (N * up,), complex)
    out[..., :h] = E[..., :h]
    out[..., N * up - (N - h):] = E[..., h:]
    return out


def integrated_power(grid, E_pix, phase=None, up=4):
    """Spatially integrated P(t) [W] of E_pix (npix, Nt) after the spectral phase `phase`
    (array on grid.Omega). Returns sorted t [fs] and P [W], time step dt/up."""
    E = E_pix if phase is None else E_pix * np.exp(1j * phase)[None, :]
    A = grid.hfft.ift_t(_pad(E, up)) * up
    P = np.sum(np.abs(A) ** 2, axis=0) * grid.dA
    t = np.fft.fftfreq(grid.Nt * up, 1.0 / (grid.Nt * grid.dt))
    return np.fft.fftshift(t) / FS, np.fft.fftshift(P)


def fourier_limit(grid, E_pix, up=4):
    """Fourier limit of the spatially integrated spectrum (the 'Fourier limit' quoted in papers)."""
    S = np.sum(np.abs(E_pix) ** 2, axis=0)
    return integrated_power(grid, np.sqrt(S)[None, :], None, up)


def fwhm(x, y):
    """FWHM of the main peak (linear interpolation)."""
    y = np.asarray(y, float)
    i = int(np.argmax(y)); h = y[i] / 2
    l, r = i, i
    while l > 0 and y[l] > h: l -= 1
    while r < len(y) - 1 and y[r] > h: r += 1
    xl = np.interp(h, [y[l], y[l + 1]], [x[l], x[l + 1]])
    xr = np.interp(h, [y[r], y[r - 1]], [x[r], x[r - 1]])
    return xr - xl


def span(lam, S, level_db=-20):
    """(lambda_min, lambda_max) [nm] of the outermost points above level_db relative to the peak."""
    ok = np.nonzero(S / S.max() >= 10 ** (level_db / 10))[0]
    return lam[ok[0]] * 1e9, lam[ok[-1]] * 1e9


def homogeneity_1d(S, rel=np.exp(-2)):
    """Spectral homogeneity as defined by Weitenberg et al. (Opt. Express 25, 20502, 2017) and used in
    the papers: V(x) = (int sqrt(S S_ref))^2 / (int S int S_ref), S_ref = spectrum at the intensity
    maximum. S[x, lambda]. Returns V(x) and its intensity-weighted mean inside the 1/e^2 width."""
    prof = S.sum(axis=1)
    S0 = S[np.argmax(prof)]
    V = np.sum(np.sqrt(S * S0[None, :]), axis=1) ** 2 / np.maximum(S.sum(axis=1) * S0.sum(), 1e-300)
    inside = prof > rel * prof.max()
    return V, float(np.sum(V[inside] * prof[inside]) / np.sum(prof[inside]))


def load_digitized(path):
    """Optional measured curve digitised from a paper figure: CSV with 2 columns (x, y)."""
    if not os.path.exists(path):
        print(f"(no {path}: digitise the figure, e.g. with WebPlotDigitizer, to overlay the measurement)")
        return None
    d = np.loadtxt(path, delimiter=",")
    d = d[np.argsort(d[:, 0])]
    return d[:, 0], d[:, 1] / d[:, 1].max()


def table(rows):
    """rows: (quantity, simulation, paper) -> aligned text table."""
    print(f"{'quantity':42s}{'simulation':>16s}{'paper':>22s}")
    print("-" * 80)
    for q, s, p in rows:
        print(f"{q:42s}{s:>16s}{p:>22s}")


def run_tag(params, grid):
    """Short fingerprint of the parameters and grid, used to name the checkpoint."""
    s = json.dumps({k: str(v) for k, v in sorted(params.items())}) + str(grid.shape) + str(PHI_MAX) + str(QUICK)
    return hashlib.md5(s.encode()).hexdigest()[:8]

## 1. Parameters from the paper

In [ ]:
P = dict(
    lambda0=1030e-9,
    energy=994e-6,           # Müller 2021, their simulation of this stage (Grebing 2020: ~1 mJ)
    fwhm=200e-15,            # paper
    tod_fs3=-1e4,            # Müller 2021: reproduces the slight spectral asymmetry
    pressure_bar=0.7,        # paper: 700 mbar argon
    n2_1bar=1.0e-23,         # paper: n2 at atmospheric pressure
    R=0.6, N=26, k=1,        # paper: RoC 0.6 m, d = 1183 mm, 26 focal passes, re-entrant
    mirror_R=0.9995,         # paper: R > 99.95 % over 970-1130 nm
    band=(960e-9, 1140e-9),  # assumption: 50 % edges 10 nm outside the specified band
    mirror_gdd=35.0,         # fitted, within the paper's specs: |GDD| < 50 fs^2 
    compressor_gdd=-1923.0,  # paper: total GDD of the chirped-mirror compressor
    lambda_min=920e-9, lambda_max=1160e-9,   # spectral window of the grid
    injection="linear",      # "linear" (as in the paper) or "nonlinear" (Kerr-matched waist)
)
REF = dict(w0_um=150, wm_um=1250, span20_nm=120, tl_fs=27, comp_fs=31, T=0.96,
           M2="< 1.15", V="98.7 / 99.4 %")
RUN_DIR = "runs_bench1"
os.makedirs(RUN_DIR, exist_ok=True)

## 2. Gas, cell and eigenmode
The eigenmode computed by the code should match the one quoted in the paper: this checks the geometry before anything else.

In [ ]:
gas = th.Material("Ar", P["pressure_bar"], n2_1bar=P["n2_1bar"])
cell = th.MPC.herriott(R=P["R"], N=P["N"], k=P["k"], medium=gas, branch="concentric",
                       n_passes=2 if QUICK else P["N"], mirror_gdd_fs2=P["mirror_gdd"],
                       mirror_reflectivity=band_mirror(P["mirror_R"], *P["band"]))
lin = cell.linear_mode(P["lambda0"])
print(cell.report())
print(f"d = {cell.d*1e3:.1f} mm (paper 1183 mm)")
print(f"linear eigenmode: waist {lin['w0']*1e6:.0f} um (paper {REF['w0_um']}), "
      f"mirror {lin['w_mirror']*1e6:.0f} um (paper {REF['wm_um']})  [1/e^2 radii]")

## 3. Grid
The finest preset that fits in memory is used. The memory estimate is about 12 copies of one field (field, Runge–Kutta buffers, linear operator, cached propagators, mirror transfer); the host also needs several copies for the diagnostics.

In [ ]:
PRESETS = ([dict(box_factor=4, points_per_waist=3, time_window_factor=4)] if QUICK else
           [dict(box_factor=8, points_per_waist=8, time_window_factor=8),
            dict(box_factor=7, points_per_waist=7, time_window_factor=7),
            dict(box_factor=6, points_per_waist=6, time_window_factor=6)])
grid, preset = pick_grid(cell, PRESETS, device_budget_gb(), lambda0=P["lambda0"],
                         lambda_min=P["lambda_min"], lambda_max=P["lambda_max"], pulse_fwhm=P["fwhm"])
print(grid.describe())

## 4. Input pulse and injection

In [ ]:
E = th.gaussian_spectrum(grid, fwhm_fs=P["fwhm"] / FS)
E = th.add_spectral_phase(grid, E, tod_fs3=P["tod_fs3"])

probe = th.Pulse.at_focus(grid, E, th.Beam(th.gaussian(lin["w0"])), P["energy"])
mm = cell.match(probe, verbose=True)          # Kerr-lens report: P/Pc, matched waist, B estimate
w0_ref = lin["w0"] if P["injection"] == "linear" else mm["w0"]
pulse = th.Pulse.at_focus(grid, E, th.Beam(th.gaussian(w0_ref), chromatic_size="eigenmode"), P["energy"])
t, Pt = dg.temporal_power(grid, pulse.Aw)
print(f"injected waist {w0_ref*1e6:.0f} um ({P['injection']}), input peak power {Pt.max()/1e9:.2f} GW "
      f"(paper: 4.5 GW), FWHM {fwhm(t / FS, Pt):.0f} fs")

## 5. Run
Resumable: with `MAX_SECONDS` set, the run stops cleanly and writes a checkpoint; running the cell again continues it. The checkpoint name contains a fingerprint of the parameters, so changing a parameter starts a fresh run.

In [ ]:
sim = th.MPCSimulation(pulse, cell, phi_max=PHI_MAX, mode_reference=w0_ref, near_field=True)

In [ ]:
tag = run_tag(P, grid)
ckpt = os.path.join(RUN_DIR, f"checkpoint_{tag}.pkl")
if RESUME and os.path.exists(ckpt):
    sim.resume(ckpt)
    print(f"resumed from {ckpt} at pass {sim.passes_done}")
if not sim.done:
    t0 = time.time()
    sim.run(max_seconds=MAX_SECONDS)
    sim.checkpoint(ckpt)
    print(f"{time.time() - t0:.0f} s for this call")
print("finished" if sim.done else "time budget reached: run this cell again to continue")

## 6. Evolution along the cell

In [ ]:
h = sim.history
npass = np.array(h["pass"])
fig, ax = plt.subplots(1, 4, figsize=(15, 3.2), layout="constrained")
ax[0].plot(npass, np.array(h["energy"]) / h["energy"][0], "o-"); ax[0].set_title("transmission")
ax[1].plot(npass, h["B"], "o-"); ax[1].set_title("B-integral, on-axis peak (rad)")
ax[2].plot(npass, np.array(h["w_peak_x"]) * 1e6, "o-", label="x")
ax[2].plot(npass, np.array(h["w_peak_y"]) * 1e6, "s-", ms=3, label="y")
ax[2].axhline(w0_ref * 1e6, color="k", ls=":", label="injected waist")
ax[2].set_title("beam radius at the cell centre (um)"); ax[2].legend(fontsize=8)
ax[3].plot(npass, h["eta00_total"], "o-", label="HG00 fraction")
ax[3].plot(npass, h["V_near"], "s-", label="homogeneity (2D, near field)")
ax[3].plot(npass, h["M2x_mean"], "^-", label="M2 x"); ax[3].legend(fontsize=8)
for a in ax:
    a.set_xlabel("pass")

## 7. Output spectrum
To overlay the measurement, digitise Fig. 2 (upper panel) of the paper into `data/grebing2020_spectrum.csv` (wavelength in nm, linear intensity).

In [ ]:
XLIM_NM = (940, 1120)
lam, S = dg.total_spectrum(grid, sim.Aw)
lam_in, S_in = sim.lam, sim.history["spectrum"][0]
meas = load_digitized("data/grebing2020_spectrum.csv")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")
for a, log in zip(ax, (False, True)):
    a.plot(lam_in * 1e9, S_in / S_in.max(), "k:", label="input")
    a.plot(lam * 1e9, S / S.max(), label="simulation")
    if meas is not None:
        a.plot(*meas, label="measured (digitised)")
    a.set_xlim(*XLIM_NM); a.set_xlabel("wavelength (nm)")
    if log:
        a.set_yscale("log"); a.set_ylim(1e-3, 2)
ax[0].legend(fontsize=8)
lo, hi = span(lam, S, -20)
print(f"-20 dB span: {lo:.0f}-{hi:.0f} nm = {hi - lo:.0f} nm (paper ~{REF['span20_nm']} nm)")

## 8. Compression
With the paper's compressor GDD, and with the best GDD found by `dg.compress`.

In [ ]:
Epix = beam_pixels(sim.Aw)
t_tl, P_tl = fourier_limit(grid, Epix)
t_c, P_c = integrated_power(grid, Epix, gdd_phase(grid, P["compressor_gdd"]))
best = dg.compress(grid, sim.Aw, gdd_bounds_fs2=(-6000, 0))
t_b, P_b = integrated_power(grid, Epix, gdd_phase(grid, best["gdd_fs2"]))

fig, ax = plt.subplots(figsize=(7, 3.6), layout="constrained")
ax.plot(t_tl, P_tl / 1e9, "k:", label=f"Fourier limit, {fwhm(t_tl, P_tl):.1f} fs")
ax.plot(t_c, P_c / 1e9, label=f"{P['compressor_gdd']:.0f} fs$^2$ (paper), {fwhm(t_c, P_c):.1f} fs")
ax.plot(t_b, P_b / 1e9, "--", label=f"best GDD {best['gdd_fs2']:.0f} fs$^2$, {fwhm(t_b, P_b):.1f} fs")
ax.set_xlim(-150, 150); ax.set_xlabel("time (fs)"); ax.set_ylabel("power (GW)"); ax.legend(fontsize=8)

## 9. Output beam: spatio-spectral homogeneity
The papers measure the collimated beam with an imaging spectrometer along one axis, and weight the homogeneity with the beam profile. The same definition is used here (reference spectrum at the intensity maximum).

In [ ]:
# near field (collimated output beam), along x: what the papers' imaging spectrometers measure
theta, lam_t, S_th = dg.spectrogram_theta_lambda(grid, sim.Aw, per_lambda=False)
V_x, V_mean = homogeneity_1d(S_th)
prof = S_th.sum(axis=1)
inside = prof > np.exp(-2) * prof.max()

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")
ax[0].pcolormesh(lam_t * 1e9, theta * 1e3, S_th / S_th.max(), shading="auto", cmap="inferno")
ax[0].set_xlim(*XLIM_NM); ax[0].set_ylim(theta[inside].min() * 2e3, theta[inside].max() * 2e3)
ax[0].set_xlabel("wavelength (nm)"); ax[0].set_ylabel("angle (mrad)"); ax[0].set_title("output S(theta_x, lambda)")
ax[1].plot(theta * 1e3, prof / prof.max(), "k", label="profile")
ax[1].plot(theta[inside] * 1e3, V_x[inside], "g", label="homogeneity V")
ax[1].set_xlim(ax[0].get_ylim()); ax[1].set_ylim(0, 1.05); ax[1].legend(fontsize=8)
ax[1].set_xlabel("angle (mrad)"); ax[1].set_title(f"weighted homogeneity (x): {V_mean*100:.1f} %")

## 10. Summary

In [ ]:
Tcell = sim.history["energy"][-1] / sim.history["energy"][0]
table([
    ("waist / mirror spot (um, 1/e^2 radius)", f"{lin['w0']*1e6:.0f} / {lin['w_mirror']*1e6:.0f}", f"{REF['w0_um']} / {REF['wm_um']}"),
    ("-20 dB width (nm)",                      f"{hi - lo:.0f}", f"~{REF['span20_nm']}"),
    ("Fourier limit (fs)",                     f"{fwhm(t_tl, P_tl):.1f}", f"{REF['tl_fs']}"),
    (f"compressed, {P['compressor_gdd']:.0f} fs^2 (fs)", f"{fwhm(t_c, P_c):.1f}", f"{REF['comp_fs']} (autocorr.)"),
    ("compressed, best GDD (fs)",              f"{fwhm(t_b, P_b):.1f}", "-"),
    ("peak power vs Fourier limit",            f"{P_c.max()/P_tl.max()*100:.0f} %", "-"),
    ("transmission (cell only in the sim.)",   f"{Tcell*100:.1f} %", f"{REF['T']*100:.0f} % (+compr.)"),
    ("M2 x / y (spectrally weighted)",         f"{sim.history['M2x_mean'][-1]:.2f} / {sim.history['M2y_mean'][-1]:.2f}", REF["M2"]),
    ("homogeneity, weighted (x)",              f"{V_mean*100:.1f} %", REF["V"]),
    ("B-integral, on-axis peak (rad)",         f"{sim.solver.B:.1f}", "~13 (beam-averaged)"),
])

## 11. Save
`bench1_stage1_output.npz` holds the spatially integrated output spectrum and the on-axis spectral phase after the paper's compressor: it is the input of notebook 2 (second cell of Müller *et al.* 2021).

In [ ]:
sim.save(os.path.join(RUN_DIR, "run.npz"), include_field=False)

A = th.to_host(sim.Aw)
lam_s, S_out = dg.total_spectrum(grid, A)
idx = np.nonzero(grid.valid)[0]
idx = idx[np.argsort(grid.lam[idx])]                         # same order as total_spectrum
phase = np.angle(A[0, 0, :] * np.exp(1j * gdd_phase(grid, P["compressor_gdd"])))[idx]
np.savez("bench1_stage1_output.npz", lam_nm=lam_s * 1e9, S=S_out, phase=np.unwrap(phase),
         energy=sim.history["energy"][-1], compressor_gdd=P["compressor_gdd"], quick=QUICK)
print("saved bench1_stage1_output.npz")

## Checks before trusting the comparison
* **Convergence:** run again with `PHI_MAX = 0.01` and with a finer preset; the spectrum and the compressed duration should not move.
* **n₂ uncertainty:** literature values for argon scatter by about ±10 %. A ±10 % change of `n2_1bar` shows how much of a mismatch it can explain.
* **Not in the model:** the folded, off-axis geometry (astigmatism, real spot pattern), the windows, the measured input spectrum and phase, and the autocorrelation deconvolution of the 31 fs.